# Tutorial 3: Iterative refinement — H_2 = JLS(G + H_1)

The JLS construction is run once on the original graph G, producing
H_1. What if we run it AGAIN on the augmented graph G + H_1?

The result H_2 = JLS(G + H_1) is:
- **A subset of H_1 on random DAGs and similar classes** (the
  shortcuts JLS adds on the augmented graph are a subset of those
  it added on the original). The shortcuts in H_1 \ H_2 are
  "self-redundant": JLS added them, but given that H_1 is already
  in the graph, it wouldn't add them again.
- **Incomparable with H_1 on the Petersen graph**: the
  iteration moves shortcuts around (different paths in the
  augmented graph reach different vertices with the new hopbound).

This notebook makes the distinction visible.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from reachq.core.algorithm import build_shortcut_set_for_reachability
from reachq.generators import petersen_graph, random_dag
from reachq.research.iterate import iterative_shortcut_set

In [ ]:
# Test 1: random DAG — H_2 ⊂ H_1
g = random_dag(n=40, edge_probability=0.3, random_seed=42)
H_1, _ = build_shortcut_set_for_reachability(
    g, omega=3.0, random_seed=42, sparsify_shortcuts=False
)
H_2 = iterative_shortcut_set(g, omega=3.0, max_iterations=3, random_seed=42)

print("random DAG n=40 p=0.3:")
print(f"  |H_1| = {len(H_1)}")
print(f"  |H_2| = {len(H_2)}")
print(f"  |H_1 ∩ H_2| = {len(H_1 & H_2)}")
print(rf"  |H_1 \ H_2| (self-redundant) = {len(H_1 - H_2)}")
print(rf"  |H_2 \ H_1| (new in iter 2) = {len(H_2 - H_1)}")
print(f"  H_2 ⊂ H_1? {H_2.issubset(H_1)}")

In [ ]:
# Test 2: Petersen graph — H_2 incomparable with H_1
g = petersen_graph()
H_1, _ = build_shortcut_set_for_reachability(
    g, omega=3.0, random_seed=42, sparsify_shortcuts=False
)
H_2 = iterative_shortcut_set(g, omega=3.0, max_iterations=3, random_seed=42)

print("Petersen (n=10):")
print(f"  |H_1| = {len(H_1)}")
print(f"  |H_2| = {len(H_2)}")
print(f"  |H_1 ∩ H_2| = {len(H_1 & H_2)}")
print(rf"  |H_1 \ H_2| = {len(H_1 - H_2)}")
print(rf"  |H_2 \ H_1| = {len(H_2 - H_1)}")
print(f"  H_2 ⊂ H_1? {H_2.issubset(H_1)}")
print(f"  H_1 ⊂ H_2? {H_1.issubset(H_2)}")

In [ ]:
# Test 3: Convergence — multiple iterations
import matplotlib.pyplot as plt

g = random_dag(n=30, edge_probability=0.3, random_seed=42)
sizes = []
H, _ = build_shortcut_set_for_reachability(
    g, omega=3.0, random_seed=42, sparsify_shortcuts=False
)
sizes.append(len(H))
core = H
for k in range(1, 5):
    H_new = iterative_shortcut_set(g, omega=3.0, max_iterations=k, random_seed=42)
    if H_new != core:
        core = core & H_new
        sizes.append(len(H_new))
    else:
        sizes.append(len(H_new))
        break

plt.figure(figsize=(8, 4))
plt.plot(range(len(sizes)), sizes, "o-")
plt.xlabel("iteration")
plt.ylabel("|H|")
plt.title("Iterative refinement: |H| vs iteration count (random DAG n=30 p=0.3)")
plt.grid(True)
plt.show()

## What you should see

1. **Random DAG**: H_2 ⊂ H_1 strictly. The iteration adds no new
   shortcuts beyond those JLS added in H_1. The shortcuts in
   H_1 \ H_2 are "self-redundant": JLS added them but, given
   H_1 already in the graph, it wouldn't re-add them.

2. **Petersen graph**: H_1 and H_2 are *incomparable*. The
   iteration moves shortcuts around. The robust core H_1 ∩ H_2
   is the union of shortcuts that BOTH iterations agree are
   needed.

3. **Convergence**: the iteration converges in 1–2 steps. The
   size of H_2 is typically smaller than H_1 on random DAGs
   (because of the self-redundancy removal). On Petersen the
   sizes are similar.

## What's next

Tutorial 4 verifies the generator output against known spectra
(Petersen, Paley, Shrikhande, Hamming) using the spectrum
helper.